# Estimación de Información Mutua

La información mutua I(X;Y) entre variables continuas no tiene forma cerrada
en general. Este cuaderno implementa estimadores no paramétricos basados en
binning y en k vecinos más cercanos (kNN), ilustrando el artículo
[Información en el Aprendizaje Estadístico](../../05-conexiones-y-aplicaciones/05-informacion-en-aprendizaje.md).

Solo usamos la biblioteca estándar de Python.

In [ ]:
import math
import random
from collections import defaultdict

## Utilidades

In [ ]:
def sample_gaussian(mu=0.0, sigma=1.0):
    """Muestrea N(mu, sigma^2) usando Box-Muller."""
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2 * math.log(u1 + 1e-15)) * math.cos(2 * math.pi * u2)
    return mu + sigma * z


def sample_bivariate_gaussian(rho, n):
    """Muestrea n pares (X,Y) con correlación rho, ambas N(0,1).
    
    I(X;Y) = -0.5 * log2(1 - rho^2) para gaussianas estándar bivariadas.
    """
    samples = []
    for _ in range(n):
        x = sample_gaussian()
        eps = sample_gaussian()
        y = rho * x + math.sqrt(1 - rho**2) * eps
        samples.append((x, y))
    return samples


def mi_gaussiana_teorica(rho):
    """Valor teórico I(X;Y) en bits para gaussiana bivariada con correlación rho."""
    if abs(rho) >= 1.0:
        return float('inf')
    return -0.5 * math.log2(1 - rho**2)


print("Información mutua teórica para distintas correlaciones:")
for rho in [0.0, 0.3, 0.5, 0.7, 0.9, 0.95]:
    print(f"  rho={rho:.2f} → I(X;Y) = {mi_gaussiana_teorica(rho):.4f} bits")

## Ejemplo 1: Estimador por binning (histograma 2D)

Discretizamos X e Y en bins y estimamos I(X;Y) = H(X) + H(Y) - H(X,Y)
a partir de frecuencias de histograma.

In [ ]:
def mi_binning(samples, n_bins=10):
    """Estima I(X;Y) por histograma 2D.
    
    Args:
        samples: lista de (x, y)
        n_bins: número de bins por dimensión
    
    Returns:
        estimación de I(X;Y) en bits
    """
    xs = [s[0] for s in samples]
    ys = [s[1] for s in samples]
    n = len(samples)
    
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    
    # Pequeño margen para incluir el máximo
    x_range = (x_max - x_min) * (1 + 1e-9)
    y_range = (y_max - y_min) * (1 + 1e-9)
    
    # Contar frecuencias conjuntas
    joint = defaultdict(int)
    marginal_x = defaultdict(int)
    marginal_y = defaultdict(int)
    
    for x, y in samples:
        bx = int((x - x_min) / x_range * n_bins)
        by = int((y - y_min) / y_range * n_bins)
        joint[(bx, by)] += 1
        marginal_x[bx] += 1
        marginal_y[by] += 1
    
    # I(X;Y) = sum_{x,y} p(x,y) log2(p(x,y)/(p(x)*p(y)))
    mi = 0.0
    for (bx, by), count in joint.items():
        pxy = count / n
        px = marginal_x[bx] / n
        py = marginal_y[by] / n
        mi += pxy * math.log2(pxy / (px * py))
    
    return mi


print("Estimador por binning (n=1000, 10 bins):")
print(f"{'rho':>6} | {'I_teorica':>10} | {'I_binning':>10} | {'error':>8}")
print("-" * 42)
random.seed(42)
for rho in [0.0, 0.3, 0.5, 0.7, 0.9]:
    samples = sample_bivariate_gaussian(rho, 1000)
    i_teorica = mi_gaussiana_teorica(rho)
    i_est = mi_binning(samples, n_bins=12)
    print(f"{rho:>6.2f} | {i_teorica:>10.4f} | {i_est:>10.4f} | {abs(i_est-i_teorica):>8.4f}")

## Ejemplo 2: Estimador kNN (Kraskov-Stögbauer-Grassberger)

El estimador KSG usa distancias al k-ésimo vecino más cercano:

$$\hat{I}(X;Y) = \psi(k) - \langle \psi(n_x) + \psi(n_y) \rangle + \psi(n)$$

donde ψ es la función digamma y n_x, n_y son contadores de vecinos
en las proyecciones marginales.

In [ ]:
def digamma(x):
    """Aproximación de la función digamma ψ(x) = d/dx log Γ(x)."""
    # Asymptotica: ψ(x) ≈ log(x) - 1/(2x) para x grande
    # Para x pequeño usamos la recurrencia ψ(x+1) = ψ(x) + 1/x
    if x < 6:
        return digamma(x + 1) - 1.0 / x
    return math.log(x) - 1.0/(2*x) - 1.0/(12*x**2) + 1.0/(120*x**4)


def chebyshev_dist_2d(p, q):
    """Distancia de Chebyshev (L∞) entre puntos 2D."""
    return max(abs(p[0]-q[0]), abs(p[1]-q[1]))


def mi_ksg(samples, k=3):
    """Estimador KSG de I(X;Y) (versión 1 de Kraskov et al. 2004).
    
    Usa distancia L∞ en el espacio conjunto y cuenta vecinos
    en las proyecciones marginales con la misma distancia.
    """
    n = len(samples)
    xs = [(s[0],) for s in samples]
    ys = [(s[1],) for s in samples]
    
    sum_digamma = 0.0
    
    for i in range(n):
        # Distancias en espacio conjunto (L∞)
        dists_joint = sorted(
            chebyshev_dist_2d(samples[i], samples[j])
            for j in range(n) if j != i
        )
        # Radio del k-ésimo vecino
        eps = dists_joint[k - 1]
        
        # Contar vecinos en marginales dentro del radio eps
        nx = sum(1 for j in range(n) if j != i and abs(xs[i][0] - xs[j][0]) < eps)
        ny = sum(1 for j in range(n) if j != i and abs(ys[i][0] - ys[j][0]) < eps)
        
        sum_digamma += digamma(nx + 1) + digamma(ny + 1)
    
    mi = digamma(k) - sum_digamma / n + digamma(n)
    # Convertir de nats a bits
    return max(0.0, mi / math.log(2))


print("Estimador KSG (n=300, k=3):")
print(f"{'rho':>6} | {'I_teorica':>10} | {'I_ksg':>10} | {'error':>8}")
print("-" * 42)
random.seed(42)
for rho in [0.0, 0.3, 0.5, 0.7, 0.9]:
    samples = sample_bivariate_gaussian(rho, 300)
    i_teorica = mi_gaussiana_teorica(rho)
    i_est = mi_ksg(samples, k=3)
    print(f"{rho:>6.2f} | {i_teorica:>10.4f} | {i_est:>10.4f} | {abs(i_est-i_teorica):>8.4f}")

## Ejemplo 3: Convergencia con el tamaño muestral

In [ ]:
rho = 0.7
i_teorica = mi_gaussiana_teorica(rho)
sample_sizes = [50, 100, 200, 500, 1000]

print(f"Convergencia para rho={rho} (I_teorica={i_teorica:.4f} bits)")
print(f"{'n':>6} | {'I_binning':>10} | {'I_ksg':>8}")
print("-" * 32)

random.seed(7)
for n in sample_sizes:
    samples = sample_bivariate_gaussian(rho, n)
    i_bin = mi_binning(samples, n_bins=max(5, int(n**0.4)))
    i_ksg = mi_ksg(samples, k=3)
    print(f"{n:>6} | {i_bin:>10.4f} | {i_ksg:>8.4f}")

## Ejemplo 4: Información mutua entre variables categóricas

Para distribuciones discretas, I(X;Y) puede calcularse exactamente
desde las frecuencias empíricas.

In [ ]:
def mi_discreta(joint_probs):
    """I(X;Y) para distribución conjunta discreta.
    
    joint_probs: diccionario {(x,y): prob}
    """
    marginal_x = defaultdict(float)
    marginal_y = defaultdict(float)
    for (x, y), p in joint_probs.items():
        marginal_x[x] += p
        marginal_y[y] += p
    
    mi = 0.0
    for (x, y), pxy in joint_probs.items():
        if pxy > 0:
            px = marginal_x[x]
            py = marginal_y[y]
            mi += pxy * math.log2(pxy / (px * py))
    return mi


# Canal BSC con volteo p: X entrada, Y salida
def mi_bsc(p):
    """I(X;Y) para canal BSC con prob. volteo p, entrada uniforme."""
    # P(X=0)=P(X=1)=1/2
    joint = {
        (0, 0): 0.5 * (1 - p),
        (0, 1): 0.5 * p,
        (1, 0): 0.5 * p,
        (1, 1): 0.5 * (1 - p),
    }
    return mi_discreta(joint)


def h_binary(p):
    if p <= 0 or p >= 1:
        return 0.0
    return -p * math.log2(p) - (1-p) * math.log2(1-p)


print("Capacidad del BSC = I(X;Y) con entrada uniforme")
print(f"{'p_flip':>8} | {'I(X;Y)':>8} | {'C=1-Hb(p)':>10}")
print("-" * 34)
for p in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    mi = mi_bsc(p) if p > 0 else 1.0
    c = 1.0 - h_binary(p)
    print(f"{p:>8.2f} | {mi:>8.4f} | {c:>10.4f}")

## Ideas clave

- I(X;Y) para variables continuas no tiene forma cerrada en general:
  se necesitan estimadores no paramétricos.
- El estimador por **binning** es simple pero sesgado: más bins → más varianza;
  menos bins → más sesgo.
- El estimador **KSG** basado en k-NN es más preciso y converge más rápido
  con el tamaño muestral.
- Para distribuciones gaussianas bivariadas, I(X;Y) = −½ log₂(1−ρ²),
  que sirve como referencia para validar los estimadores.

## Ejercicios

1. Implementa el estimador de entropía diferencial por binning:
   h(X) ≈ -∑ p_i log p_i + log(Δ), donde Δ es el ancho de bin.
   Verifica con X ~ N(0,1): h_teorica = ½ log(2πe) ≈ 2.047 bits.
2. ¿Cómo cambia el error del estimador KSG al variar k (1, 3, 5, 10)?
   Experimenta con rho=0.5, n=500.
3. Genera muestras de X y Y=X² donde X ~ Uniforme(−2, 2). Calcula
   I_binning(X,Y). ¿Por qué es alta aunque Y es determinista de X?
4. En el cuello de botella informacional se minimiza I(X;T) preservando
   I(T;Y). Simula un escenario donde X = (mensaje relevante, ruido) y
   Y = mensaje relevante. ¿Cuál es el T óptimo?